# Week #4: Sorting a World of Bars

[Open this notebook in Colab](https://colab.research.google.com/github/mmoffitt/engineering-intelligence/blob/main/notebooks/week04.ipynb)

> **Important:** Before you do any work, save your own copy to Google Drive.
>
> In Colab: **File → Save a copy in Drive**

In Week 3, you inferred several transformations from examples. This week, you will stay with one ARC-style task and build a complete solver one piece at a time.

The central idea is a powerful one:

> **grid → objects → transformed objects → grid**

## Goals

By the end of this notebook, you will be able to:

- infer the rule of a new ARC-style task
- write a function that sorts a list
- detect objects in a grid and describe their properties
- rebuild a grid from object descriptions
- combine several tested helpers into one solver
- explain the assumptions that make your solver work

## Setup

Run the next cell once. It provides grid drawings and the **Check Exercise** buttons. If the runtime restarts, run this cell again.

In [ ]:
#@title Setup utilities (run once)

import copy
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from matplotlib.colors import ListedColormap

GRID_COLORS = [
    "#000000", "#0074D9", "#FF4136", "#2ECC40", "#FFDC00",
    "#AAAAAA", "#F012BE", "#FF851B", "#7FDBFF", "#870C25"
]
GRID_CMAP = ListedColormap(GRID_COLORS)

def draw_grid(ax, grid, title=""):
    rows = len(grid)
    cols = len(grid[0]) if rows else 0
    ax.imshow(grid, cmap=GRID_CMAP, vmin=0, vmax=9)
    font_size = max(6, min(12, 120 // max(rows, cols, 1)))
    colors_with_white_text = {0, 1, 2, 6, 9}
    for row_index, row in enumerate(grid):
        for column_index, color in enumerate(row):
            text_color = "white" if color in colors_with_white_text else "black"
            ax.text(column_index, row_index, str(color), ha="center", va="center",
                    color=text_color, fontsize=font_size, fontweight="bold")
    ax.set_title(title)
    ax.set_xticks([x - 0.5 for x in range(1, cols)], minor=True)
    ax.set_yticks([y - 0.5 for y in range(1, rows)], minor=True)
    ax.grid(which="minor", color="white", linewidth=1)
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

def show_grid(grid, title="Grid"):
    _, ax = plt.subplots(figsize=(4, 3))
    draw_grid(ax, grid, title)
    plt.show()

def show_pair(input_grid, output_grid, label=""):
    _, axes = plt.subplots(1, 2, figsize=(8, 3))
    draw_grid(axes[0], input_grid, "Input")
    draw_grid(axes[1], output_grid, "Output")
    if label:
        plt.suptitle(label)
    plt.tight_layout()
    plt.show()

def show_examples(examples):
    for number, example in enumerate(examples, start=1):
        show_pair(example["input"], example["output"], f"Training example {number}")

def make_check_button(check_fn):
    button = widgets.Button(description="Check Exercise", button_style="info")
    output = widgets.Output()
    def on_click(_):
        output.clear_output()
        with output:
            check_fn()
    button.on_click(on_click)
    display(button, output)

print("Setup complete.")

# The task

We created this ARC-style task for the internship. It is not from the official ARC dataset.

Study all three training examples before reading further. Each colored vertical bar is an object. The black cells form the background.

In [ ]:
training_examples = [
    {
        "input": [
            [0,0,0,0,0,0,0,0,0],
            [0,0,0,0,0,1,0,0,0],
            [0,0,0,0,0,1,0,0,0],
            [0,2,0,0,0,1,0,0,0],
            [0,2,0,0,0,1,0,3,0],
            [0,2,0,4,0,1,0,3,0],
        ],
        "output": [
            [0,0,0,0,0,0,0,0,0],
            [0,0,0,0,0,0,0,1,0],
            [0,0,0,0,0,0,0,1,0],
            [0,0,0,0,0,2,0,1,0],
            [0,0,0,3,0,2,0,1,0],
            [0,4,0,3,0,2,0,1,0],
        ],
    },
    {
        "input": [
            [0,0,0,6,0,0,0],
            [0,8,0,6,0,0,0],
            [0,8,0,6,0,0,0],
            [0,8,0,6,0,0,0],
            [0,8,0,6,0,2,0],
        ],
        "output": [
            [0,0,0,0,0,6,0],
            [0,0,0,8,0,6,0],
            [0,0,0,8,0,6,0],
            [0,0,0,8,0,6,0],
            [0,2,0,8,0,6,0],
        ],
    },
    {
        "input": [
            [0,0,0,7,0,0,0,0,0,0,0],
            [0,0,0,7,0,0,0,0,0,8,0],
            [0,0,0,7,0,0,0,4,0,8,0],
            [0,0,0,7,0,0,0,4,0,8,0],
            [0,1,0,7,0,0,0,4,0,8,0],
            [0,1,0,7,0,2,0,4,0,8,0],
            [0,1,0,7,0,2,0,4,0,8,0],
        ],
        "output": [
            [0,0,0,0,0,0,0,0,0,7,0],
            [0,0,0,0,0,0,0,8,0,7,0],
            [0,0,0,0,0,4,0,8,0,7,0],
            [0,0,0,0,0,4,0,8,0,7,0],
            [0,0,0,1,0,4,0,8,0,7,0],
            [0,2,0,1,0,4,0,8,0,7,0],
            [0,2,0,1,0,4,0,8,0,7,0],
        ],
    },
]

show_examples(training_examples)

## Observation notebook

Answer these before coding:

1. What property determines the new left-to-right order?
2. What happens to each bar's color?
3. What happens to each bar's height?
4. Do the grid dimensions change?
5. Where does the bottom of every bar remain?
6. What pattern do the black columns follow?

### My rule

- **The bars are ordered by:**
- **The colors:**
- **The heights:**
- **The grid dimensions:**
- **My rule in one sentence:**

## The task's format

Here are the assumptions our invented task follows:

- `0` is the black background.
- Every bar is one cell wide, solid-colored, and bottom-aligned.
- A one-column black gap separates neighboring bars.
- A one-column black border appears at the left and right.
- Every bar has a different height.
- The output arranges the same bars from **shortest to tallest**, left to right.

Stating assumptions matters. A solver can be correct for a clearly defined family of grids without solving every imaginable bar puzzle.

# Part 1: Learn to sort

Before sorting bars, practice sorting plain numbers. We will build the process one small step at a time.

## Exercise 1: Find the smallest value

Write `find_smallest(numbers)` so it returns the smallest number in a nonempty list. Start with the first number, then use a loop to compare it with every other number.

In [ ]:
def find_smallest(numbers):
    smallest = numbers[0]
    # Compare every number with smallest.
    return smallest

In [ ]:
#@title Check Exercise 1

def check_exercise_1():
    tests = [([4, 1, 5, 2], 1), ([9, 7, 8], 7), ([3], 3), ([5, -2, 0], -2)]
    try:
        for numbers, expected in tests:
            if find_smallest(numbers) != expected:
                print("Not yet. Compare each value with the smallest value seen so far.")
                return
    except Exception as exc:
        print("Your function raised an error:", exc)
        return
    print("Exercise 1 complete: you can find the smallest value!")

make_check_button(check_exercise_1)

## Exercise 2: Sort a list

Write `sort_numbers(numbers)` using this plan:

1. Copy the input with `remaining = numbers[:]`.
2. Repeat once for every input number.
3. Find the smallest value remaining.
4. Append it to `result`.
5. Remove it from `remaining` with `remaining.remove(smallest)`.

Return a new list and leave the input unchanged.

In [ ]:
def sort_numbers(numbers):
    remaining = numbers[:]
    result = []
    # Repeatedly move the smallest remaining number into result.
    return result

In [ ]:
#@title Check Exercise 2

def check_exercise_2():
    tests = [([4, 1, 5, 2], [1, 2, 4, 5]), ([3, 3, 1], [1, 3, 3]), ([7], [7]), ([], [])]
    try:
        for numbers, expected in tests:
            original = numbers[:]
            got = sort_numbers(numbers)
            if got != expected:
                print("Not yet. Repeatedly append the smallest remaining value.")
                print("Got:", got, "Expected:", expected)
                return
            if numbers != original:
                print("The values are sorted, but the input list changed. Work on a copy.")
                return
    except Exception as exc:
        print("Your function raised an error:", exc)
        return
    print("Exercise 2 complete: your sorting function works!")

make_check_button(check_exercise_2)

# Part 2: Turn pixels into objects

A useful ARC solver often converts many individual cells into a smaller collection of meaningful objects. We will describe each bar with a dictionary:

```python
{"color": 2, "height": 4}
```

The dictionary records what matters and leaves out the background cells.

## Exercise 3: Measure one bar

Write `measure_bar(grid, column_index)`. The selected column will contain exactly one solid bar. Return a dictionary containing its `"color"` and `"height"`.

Hint: scan every row. Each nonzero cell adds one to the height and reveals the bar's color.

In [ ]:
def measure_bar(grid, column_index):
    color = 0
    height = 0
    # Inspect the cell in column_index of every row.
    return {"color": color, "height": height}

In [ ]:
#@title Check Exercise 3

def check_exercise_3():
    grid = [[0,0,0,5,0], [0,2,0,5,0], [0,2,0,5,0], [0,2,0,5,0]]
    tests = [(1, {"color": 2, "height": 3}), (3, {"color": 5, "height": 4})]
    try:
        for column_index, expected in tests:
            got = measure_bar(grid, column_index)
            if got != expected:
                print("Not yet. Count nonzero cells and record their color.")
                print("Got:", got, "Expected:", expected)
                return
    except Exception as exc:
        print("Your function raised an error:", exc)
        return
    print("Exercise 3 complete: pixels became bar descriptions!")

make_check_button(check_exercise_3)

## Exercise 4: Find every bar

Write `find_bars(grid)` so it returns the bar dictionaries from left to right.

Because every bar reaches the bottom row, a column contains a bar when its bottom cell is nonzero. For each such column, call `measure_bar`.

In [ ]:
def find_bars(grid):
    bars = []
    bottom_row = grid[-1]
    # Inspect each column in bottom_row.
    return bars

In [ ]:
#@title Check Exercise 4

def check_exercise_4():
    tests = [
        (training_examples[0]["input"], [{"color":2,"height":3}, {"color":4,"height":1}, {"color":1,"height":5}, {"color":3,"height":2}]),
        (training_examples[1]["input"], [{"color":8,"height":4}, {"color":6,"height":5}, {"color":2,"height":1}]),
    ]
    try:
        for grid, expected in tests:
            got = find_bars(grid)
            if got != expected:
                print("Not yet. Scan bottom-row columns from left to right.")
                print("Got:     ", got)
                print("Expected:", expected)
                return
    except Exception as exc:
        print("Your function raised an error:", exc)
        return
    print("Exercise 4 complete: you found every bar!")

make_check_button(check_exercise_4)

# Part 3: Sort objects

Sorting bar dictionaries is similar to sorting numbers, except that comparisons use each bar's `"height"`.

## Exercise 5: Sort the bars

Write `sort_bars(bars)` so it returns a new list ordered from shortest to tallest. Preserve every complete dictionary so each height stays paired with its original color.

You can adapt your strategy from `sort_numbers`: repeatedly search `remaining` for the bar with the smallest height, append it to `result`, and remove it.

In [ ]:
def sort_bars(bars):
    remaining = bars[:]
    result = []
    # Compare bar["height"] values.
    return result

In [ ]:
#@title Check Exercise 5

def check_exercise_5():
    tests = [
        ([{"color":2,"height":4}, {"color":7,"height":1}, {"color":3,"height":2}],
         [{"color":7,"height":1}, {"color":3,"height":2}, {"color":2,"height":4}]),
        ([{"color":9,"height":3}], [{"color":9,"height":3}]),
    ]
    try:
        for bars, expected in tests:
            original = copy.deepcopy(bars)
            got = sort_bars(bars)
            if got != expected:
                print("Not yet. Compare heights, but move each complete bar dictionary.")
                print("Got:     ", got)
                print("Expected:", expected)
                return
            if bars != original:
                print("The order is right, but the input list changed.")
                return
    except Exception as exc:
        print("Your function raised an error:", exc)
        return
    print("Exercise 5 complete: the bar objects are sorted!")

make_check_button(check_exercise_5)

# Part 4: Turn objects back into pixels

Now reverse the representation. Given a list of bars, generate a new grid with a black border, black gaps, and bottom-aligned bars.

## Exercise 6: Draw the bars

Write `draw_bars(bars, grid_height)`.

Use this plan:

1. The width is `2 * len(bars) + 1`.
2. Build a black grid of the requested height and calculated width.
3. Bar number `0` belongs in column `1`, bar number `1` in column `3`, and so on. The formula is `2 * bar_index + 1`.
4. Fill each bar upward from the bottom using its color and height.

In [ ]:
def draw_bars(bars, grid_height):
    grid_width = 2 * len(bars) + 1
    grid = []
    # First build grid_height black rows.

    # Then draw each bar upward from the bottom.
    return grid

In [ ]:
#@title Check Exercise 6

def check_exercise_6():
    bars = [{"color":4,"height":1}, {"color":3,"height":2}, {"color":2,"height":3}]
    expected = [[0,0,0,0,0,2,0], [0,0,0,0,0,2,0], [0,4,0,3,0,2,0]]
    try:
        got = draw_bars(bars, 3)
        if got != expected:
            print("Not yet. Check the grid dimensions, bar columns, and bottom alignment.")
            print("Got:     ", got)
            print("Expected:", expected)
            return
        if draw_bars([], 2) != [[0], [0]]:
            print("Check what your generator does when there are no bars.")
            return
    except Exception as exc:
        print("Your function raised an error:", exc)
        return
    print("Exercise 6 complete: object descriptions became a grid!")

make_check_button(check_exercise_6)

# Part 5: Put everything together

Each helper now has one clear responsibility:

```text
input grid → find_bars → sort_bars → draw_bars → output grid
```

A `solve` function can connect them without reimplementing their logic.

## Final exercise: Write the solver

Write `solve(grid)` using the three helpers. The output height is the same as the input height: `len(grid)`.

In [ ]:
def solve(grid):
    # Parse, sort, and generate.
    return None

In [ ]:
#@title Check the complete solver

def check_solver():
    try:
        for number, example in enumerate(training_examples, start=1):
            original = copy.deepcopy(example["input"])
            got = solve(example["input"])
            if got != example["output"]:
                print(f"Training example {number} does not match yet.")
                return
            if example["input"] != original:
                print("The answer is right, but solve changed its input.")
                return
        hidden_input = [
            [0,6,0,0,0,0,0,0,0],
            [0,6,0,0,0,0,0,0,0],
            [0,6,0,0,0,8,0,0,0],
            [0,6,0,3,0,8,0,0,0],
            [0,6,0,3,0,8,0,0,0],
            [0,6,0,3,0,8,0,1,0],
            [0,6,0,3,0,8,0,1,0],
        ]
        hidden_expected = [
            [0,0,0,0,0,0,0,6,0],
            [0,0,0,0,0,0,0,6,0],
            [0,0,0,0,0,8,0,6,0],
            [0,0,0,3,0,8,0,6,0],
            [0,0,0,3,0,8,0,6,0],
            [0,1,0,3,0,8,0,6,0],
            [0,1,0,3,0,8,0,6,0],
        ]
        got = solve(hidden_input)
        if got != hidden_expected:
            print("The training examples pass, but a new test grid does not.")
            return
    except Exception as exc:
        print("Your solver raised an error:", exc)
        return
    print("All checks pass. You built a complete ARC-style solver!")

make_check_button(check_solver)

## Apply the solver to a test input

This test grid was not one of the training pairs. Run the cell to see only its input. Before revealing your program's result, predict the left-to-right color order on paper.

In [ ]:
test_input = [
    [0,0,0,7,0,0,0,0,0],
    [0,2,0,7,0,0,0,0,0],
    [0,2,0,7,0,0,0,0,0],
    [0,2,0,7,0,4,0,0,0],
    [0,2,0,7,0,4,0,0,0],
    [0,2,0,7,0,4,0,9,0],
]

show_grid(test_input, "Test input")

### My prediction

- **Bar colors from shortest to tallest:**
- **Bar heights from shortest to tallest:**

In [ ]:
test_output = solve(test_input)
show_pair(test_input, test_output, "Your solver's answer")

# Stretch challenges

Choose any that interest you:

1. **Reverse the order:** Add a parameter that can arrange bars tallest-to-shortest.
2. **Equal heights:** Decide on a clear tie-breaking rule, create examples, and update the solver.
3. **Wider bars:** Change the parser and generator so bars may be more than one cell wide.
4. **Messier spacing:** Detect bars even when the black gaps or borders have different widths.
5. **Make a puzzle generator:** Generate valid random inputs, solve them, and display the resulting training pairs.

## Wrap-up and reflection

This week, one transformation became a complete program made of smaller, testable parts. Answer briefly:

- Why was a bar dictionary easier to work with than individual pixels?
- Which helper was hardest to write?
- What assumption would break your current solver?
- Why test each helper separately before testing `solve`?

### My reflection

- **Objects helped because:**
- **Hardest helper:**
- **One important assumption:**
- **Why separate tests helped:**

## Share your completed notebook

When you finish:

1. Confirm that all six exercise checks and the complete solver check pass.
2. Complete the reflection.
3. In Colab, click **Share**.
4. Add **michael.d.moffitt@gmail.com** as a Viewer or Commenter.
5. Send the share invite.

Building the complete core solver is the goal. The stretch challenges are optional.